# Integración de Datos 2019-2020 al Dataset Maestro

**Objetivo:** Extender el dataset maestro (`master_dataset_fase2_multivariado.csv`, 2021-2025) con datos de 2019-2020 de las tres fuentes: MIDAGRI, NASA POWER e INDECI.

**Nota importante:** El master existente contiene valores **normalizados** (z-scores). Este notebook trabaja con datos crudos para 2019-2020 y al final concatena + re-normaliza todo el conjunto extendido.

| Paso | Fuente | Archivo |
|------|--------|---------|
| 1 | MIDAGRI (Sisagri) | `sources/midagri/Sisagri_2016_2025.xlsx` hoja `2016_2020` |
| 2 | NASA POWER | `data/interim/nasa/clima_dataset_2019_2020.csv` |
| 3 | INDECI (SINPAD) | `data/interim/indeci/indeci_temporal_2019_2025.csv` |
| 4 | Merge 3 fuentes | Cruce por departamento, provincia, año, mes |
| 5 | Concatenar con master | `data/processed/master_dataset_extendido.csv` |

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

ROOT = Path.cwd()
while not (ROOT / 'CLAUDE.md').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

print(f"Raíz del proyecto: {ROOT}")
print(f"Pandas: {pd.__version__}")

Raíz del proyecto: C:\Machine-learming\Machine-Learning-Multimodal--Agro-NLP-Clima-
Pandas: 3.0.2


In [2]:
# Cargar master existente para referencia de provincias y estructura
master_orig = pd.read_csv(ROOT / 'data/processed/master_dataset_fase2_multivariado.csv')
PROVINCIAS_MASTER = set(master_orig.groupby(['departamento', 'provincia']).size().index)

print(f"Master existente: {master_orig.shape}")
print(f"Rango temporal: {master_orig['fecha_evento'].min()} → {master_orig['fecha_evento'].max()}")
print(f"Combinaciones dpto-prov: {len(PROVINCIAS_MASTER)}")
print(f"Columnas ({len(master_orig.columns)}): {master_orig.columns.tolist()}")

Master existente: (5880, 24)
Rango temporal: 2021-01-01 → 2025-08-01
Combinaciones dpto-prov: 105
Columnas (24): ['fecha_evento', 'departamento', 'provincia', 'produccion_t', 'precio_chacra_kg', 'num_emergencias', 'total_afectados', 'hectareas_cultivo_perdidas', 'ALLSKY_SFC_SW_DWN', 'PRECTOTCORR', 'QV2M', 'RH2M', 'T2M', 'T2M_MAX', 'T2M_MIN', 'WS2M', 'lat', 'lon', 'month_sin', 'month_cos', 'mes_num', 'trimestre_num', 'trimestre_sin', 'trimestre_cos']


---
## PASO 1 — Preparar MIDAGRI 2019-2020

- Leer hoja `2016_2020` de `Sisagri_2016_2025.xlsx`
- Filtrar cultivos: solo `LIMON` y `LIMON DULCE` (excluir LIMA, PAJA LIMA)
- Filtrar años 2019 y 2020
- Agregar a nivel **provincia-mes**: suma de `PRODUCCION(t)` y media de `MTO_PRECCHAC`

In [3]:
# PASO 1: MIDAGRI 2019-2020
midagri_raw = pd.read_excel(
    ROOT / 'sources/midagri/Sisagri_2016_2025.xlsx',
    sheet_name='2016_2020'
)
print(f"MIDAGRI hoja completa: {midagri_raw.shape}")
print(f"Cultivos únicos con 'LIMON': {midagri_raw[midagri_raw['dsc_Cultivo'].str.contains('LIMON', case=False, na=False)]['dsc_Cultivo'].unique()}")

# Filtrar LIMON / LIMON DULCE, excluir LIMA y PAJA
mask_cultivo = (
    midagri_raw['dsc_Cultivo'].str.contains('LIMON', case=False, na=False)
    & ~midagri_raw['dsc_Cultivo'].str.contains('LIMA|PAJA', case=False, na=False)
)
mask_anho = midagri_raw['anho'].isin([2019, 2020])

midagri_filtered = midagri_raw[mask_cultivo & mask_anho].copy()
print(f"\nFiltrado (LIMON/LIMON DULCE, 2019-2020): {midagri_filtered.shape}")
print(f"Cultivos incluidos: {midagri_filtered['dsc_Cultivo'].unique()}")
print(f"Años: {sorted(midagri_filtered['anho'].unique())}")
print(f"Dptos: {sorted(midagri_filtered['Dpto'].unique())}")

# Agregar a nivel provincia-mes
midagri_agg = (
    midagri_filtered
    .groupby(['anho', 'mes', 'Dpto', 'Prov'])
    .agg({
        'PRODUCCION(t)': 'sum',
        'MTO_PRECCHAC (S/ x kg)': 'mean'
    })
    .reset_index()
    .rename(columns={
        'Dpto': 'departamento',
        'Prov': 'provincia',
        'PRODUCCION(t)': 'produccion_t',
        'MTO_PRECCHAC (S/ x kg)': 'precio_chacra_kg'
    })
)

# Filtrar solo las provincias que existen en el master
midagri_agg['_key'] = list(zip(midagri_agg['departamento'], midagri_agg['provincia']))
midagri_agg = midagri_agg[midagri_agg['_key'].isin(PROVINCIAS_MASTER)].drop(columns='_key')

print(f"\nAgregado provincia-mes (solo provs del master): {midagri_agg.shape}")
print(f"Combinaciones dpto-prov: {midagri_agg.groupby(['departamento','provincia']).ngroups}")
print(f"\nMuestra:")
midagri_agg.head(10)

MIDAGRI hoja completa: (818683, 10)
Cultivos únicos con 'LIMON': <ArrowStringArray>
['LIMON', 'LIMON DULCE']
Length: 2, dtype: str

Filtrado (LIMON/LIMON DULCE, 2019-2020): (8324, 10)
Cultivos incluidos: <ArrowStringArray>
['LIMON', 'LIMON DULCE']
Length: 2, dtype: str
Años: [np.int64(2019), np.int64(2020)]
Dptos: ['AMAZONAS', 'ANCASH', 'APURIMAC', 'AREQUIPA', 'AYACUCHO', 'CAJAMARCA', 'CUSCO', 'HUANCAVELICA', 'HUANUCO', 'ICA', 'JUNIN', 'LA LIBERTAD', 'LAMBAYEQUE', 'LIMA', 'LORETO', 'MADRE DE DIOS', 'MOQUEGUA', 'PASCO', 'PIURA', 'PUNO', 'SAN MARTIN', 'TUMBES', 'UCAYALI']

Agregado provincia-mes (solo provs del master): (1913, 6)
Combinaciones dpto-prov: 104

Muestra:


,anho,mes,departamento,provincia,produccion_t,precio_chacra_kg
0,2019,1,AMAZONAS,BAGUA,39.5,1.475000
1,2019,1,AMAZONAS,CHACHAPOYAS,25.0,2.850000
2,2019,1,AMAZONAS,LUYA,47.7,1.566667
3,2019,1,AMAZONAS,UTCUBAMBA,455.2,0.916667
4,2019,1,ANCASH,CARHUAZ,10.0,1.600000
5,2019,1,ANCASH,CASMA,48.0,1.587500
6,2019,1,ANCASH,HUARMEY,13.0,1.575000
7,2019,1,ANCASH,HUAYLAS,20.0,1.500000
8,2019,1,ANCASH,SANTA,13.0,1.675000
9,2019,1,ANCASH,YUNGAY,6.0,1.650000


---
## PASO 2 — Preparar NASA POWER 2019-2020

- Leer `data/interim/nasa/clima_dataset_2019_2020.csv`
- Estandarizar columna `DATE` → extraer `anho`, `mes`
- Verificar cobertura de departamentos/provincias vs master

In [4]:
# PASO 2: NASA POWER 2019-2020
nasa_raw = pd.read_csv(ROOT / 'data/interim/nasa/clima_dataset_2019_2020.csv')
print(f"NASA POWER 2019-2020: {nasa_raw.shape}")
print(f"Columnas: {nasa_raw.columns.tolist()}")

# Extraer anho y mes de DATE
nasa_raw['DATE'] = pd.to_datetime(nasa_raw['DATE'])
nasa_raw['anho'] = nasa_raw['DATE'].dt.year
nasa_raw['mes'] = nasa_raw['DATE'].dt.month

# Columnas climáticas que necesitamos (las mismas del master)
COLS_CLIMA = ['ALLSKY_SFC_SW_DWN', 'PRECTOTCORR', 'QV2M', 'RH2M',
              'T2M', 'T2M_MAX', 'T2M_MIN', 'WS2M', 'lat', 'lon']

nasa_clean = nasa_raw[['departamento', 'provincia', 'anho', 'mes'] + COLS_CLIMA].copy()

# Verificar cobertura vs master
nasa_provs = set(zip(nasa_clean['departamento'], nasa_clean['provincia']))
en_master_no_nasa = PROVINCIAS_MASTER - nasa_provs
en_nasa_no_master = nasa_provs - PROVINCIAS_MASTER

print(f"\nCobertura:")
print(f"  Provs en master: {len(PROVINCIAS_MASTER)}")
print(f"  Provs en NASA 2019-2020: {len(nasa_provs)}")
print(f"  En master pero NO en NASA: {len(en_master_no_nasa)}")
if en_master_no_nasa:
    for d, p in sorted(en_master_no_nasa):
        print(f"    - {d} / {p}")
print(f"  En NASA pero NO en master: {len(en_nasa_no_master)}")

print(f"\nRango temporal: {nasa_clean['anho'].min()}-{nasa_clean['mes'].min():02d} → {nasa_clean['anho'].max()}-{nasa_clean['mes'].max():02d}")
print(f"\nMuestra:")
nasa_clean.head(5)

NASA POWER 2019-2020: (2448, 15)
Columnas: ['departamento', 'provincia', 'DATE', 'ALLSKY_SFC_SW_DWN', 'PRECTOTCORR', 'QV2M', 'RH2M', 'T2M', 'T2M_MAX', 'T2M_MIN', 'WS2M', 'month_sin', 'month_cos', 'lat', 'lon']

Cobertura:
  Provs en master: 105
  Provs en NASA 2019-2020: 102
  En master pero NO en NASA: 5
    - ANCASH / CARHUAZ
    - ANCASH / YUNGAY
    - HUANUCO / MARAÑON
    - LA LIBERTAD / ASCOPE
    - LORETO / DATEM DEL MARAÑON
  En NASA pero NO en master: 2

Rango temporal: 2019-01 → 2020-12

Muestra:


,departamento,provincia,anho,mes,ALLSKY_SFC_SW_DWN,PRECTOTCORR,QV2M,RH2M,T2M,T2M_MAX,T2M_MIN,WS2M,lat,lon
0,AMAZONAS,BAGUA,2019,1,13.97,1.44,13.14,71.20,22.25,31.56,15.54,2.07,-5.638889,-78.531111
1,AMAZONAS,BAGUA,2019,2,12.28,1.80,13.95,73.77,22.48,30.59,16.67,1.75,-5.638889,-78.531111
2,AMAZONAS,BAGUA,2019,3,13.38,1.82,13.67,73.91,22.18,30.48,16.71,1.88,-5.638889,-78.531111
3,AMAZONAS,BAGUA,2019,4,15.18,1.25,13.59,71.92,22.59,32.92,16.24,1.78,-5.638889,-78.531111
4,AMAZONAS,BAGUA,2019,5,14.02,1.00,12.99,70.72,22.28,31.56,15.69,1.91,-5.638889,-78.531111


---
## PASO 3 — Preparar INDECI 2019-2020

- Leer `data/interim/indeci/indeci_temporal_2019_2025.csv`
- Filtrar solo años 2019-2020
- Limpiar encoding de nombres de departamento/provincia
- Verificar columnas: `num_emergencias`, `total_afectados`, `hectareas_cultivo_perdidas`

In [5]:
# PASO 3: INDECI 2019-2020
indeci_raw = pd.read_csv(ROOT / 'data/interim/indeci/indeci_temporal_2019_2025.csv')
print(f"INDECI completo: {indeci_raw.shape}")
print(f"Columnas: {indeci_raw.columns.tolist()}")
print(f"fecha_evento sample: {indeci_raw['fecha_evento'].head(3).tolist()}")

# Extraer anho y mes del formato YYYY-MM
indeci_raw['anho'] = indeci_raw['fecha_evento'].str[:4].astype(int)
indeci_raw['mes'] = indeci_raw['fecha_evento'].str[5:7].astype(int)

# Filtrar 2019-2020
indeci_filtered = indeci_raw[indeci_raw['anho'].isin([2019, 2020])].copy()
print(f"\nFiltrado 2019-2020: {indeci_filtered.shape}")
print(f"Años: {sorted(indeci_filtered['anho'].unique())}")

# Limpiar encoding de departamentos/provincias (caracteres corruptos)
ENCODING_MAP = {
    'APURA\x8dMAC': 'APURIMAC',
    'HUA\x81NUCO': 'HUANUCO',
    'JUNA\x8dN': 'JUNIN',
    'SAN MARTA\x8dN': 'SAN MARTIN',
}

for col in ['departamento', 'provincia']:
    for bad, good in ENCODING_MAP.items():
        indeci_filtered[col] = indeci_filtered[col].str.replace(bad, good, regex=False)
    # Normalizar espacios en provincias compuestas
    indeci_filtered[col] = indeci_filtered[col].str.strip().str.upper()

# Normalizar provincias que tienen formato diferente
PROV_MAP = {
    'ALTOAMAZONAS': 'ALTO AMAZONAS',
    'DATEMDELMARAÑON': 'DATEM DEL MARAÑON',
    'ANTONIORAYMONDI': 'ANTONIO RAYMONDI',
    'MARISCALRAMONCASTILLA': 'MARISCAL RAMON CASTILLA',
    'GENERALSANCHEZCERRO': 'GENERAL SANCHEZ CERRO',
    'MARISCALCACERES': 'MARISCAL CACERES',
    'LEONCIOP RADO': 'LEONCIO PRADO',
    'CONTRALMIRA NTEVILLAR': 'CONTRALMIRANTE VILLAR',
    'MARISCALNIETO': 'MARISCAL NIETO',
    'SANCHEZCARRION': 'SANCHEZ CARRION',
    'PAUCARDELSARASARA': 'PAUCAR DEL SARA SARA',
    'GRANCHIMU': 'GRAN CHIMU',
    'SANIGNACIO': 'SAN IGNACIO',
    'SANMARCOS': 'SAN MARCOS',
    'SANMIGUEL': 'SAN MIGUEL',
    'SANTACRUZ': 'SANTA CRUZ',
    'SANMARTIN': 'SAN MARTIN',
    'LACONVENCION': 'LA CONVENCION',
    'LALIBERTAD': 'LA LIBERTAD',
    'LAUNION': 'LA UNION',
    'LAMAR': 'LA MAR',
    'ELDORADO': 'EL DORADO',
    'PADREABAD': 'PADRE ABAD',
    'PUERTOINCA': 'PUERTO INCA',
    'CORONELPORTILLO': 'CORONEL PORTILLO',
    'VILCASHUAMAN': 'VILCAS HUAMAN',
    'MADREDEDI OS': 'MADRE DE DIOS',
}

for bad, good in PROV_MAP.items():
    indeci_filtered['provincia'] = indeci_filtered['provincia'].str.replace(bad, good, regex=False)

# Filtrar solo las provincias que existen en el master
COLS_INDECI = ['num_emergencias', 'total_afectados', 'hectareas_cultivo_perdidas']
indeci_clean = indeci_filtered[['departamento', 'provincia', 'anho', 'mes'] + COLS_INDECI].copy()

indeci_clean['_key'] = list(zip(indeci_clean['departamento'], indeci_clean['provincia']))
indeci_match = indeci_clean[indeci_clean['_key'].isin(PROVINCIAS_MASTER)].drop(columns='_key')
indeci_no_match = indeci_clean[~indeci_clean['_key'].isin(PROVINCIAS_MASTER)]

print(f"\nProvs INDECI que matchean con master: {indeci_match.groupby(['departamento','provincia']).ngroups}")
print(f"Provs INDECI sin match: {indeci_no_match['_key'].nunique()}")
if len(indeci_no_match) > 0:
    print("  Sin match (primeras 10):")
    for k in sorted(indeci_no_match['_key'].unique())[:10]:
        print(f"    - {k}")

print(f"\nColumnas de emergencia verificadas: {COLS_INDECI}")
print(f"Nulos en INDECI filtrado:")
print(indeci_match[COLS_INDECI].isnull().sum())
print(f"\nMuestra:")
indeci_match.head(5)

INDECI completo: (2749, 6)
Columnas: ['fecha_evento', 'departamento', 'provincia', 'num_emergencias', 'total_afectados', 'hectareas_cultivo_perdidas']
fecha_evento sample: ['2019-01', '2019-01', '2019-01']

Filtrado 2019-2020: (2749, 8)
Años: [np.int64(2019), np.int64(2020)]

Provs INDECI que matchean con master: 97
Provs INDECI sin match: 99
  Sin match (primeras 10):
    - ('AMAZONAS', 'BONGARA')
    - ('AMAZONAS', 'CONDORCANQUI')
    - ('AMAZONAS', 'RODRIGUEZDEMENDOZA')
    - ('ANCASH', 'AIJA')
    - ('ANCASH', 'ANTONIO RAYMONDI')
    - ('ANCASH', 'ASUNCION')
    - ('ANCASH', 'BOLOGNESI')
    - ('ANCASH', 'CARLOSFERMINFITZCARRALD')
    - ('ANCASH', 'CORONGO')
    - ('ANCASH', 'HUARAZ')

Columnas de emergencia verificadas: ['num_emergencias', 'total_afectados', 'hectareas_cultivo_perdidas']
Nulos en INDECI filtrado:
num_emergencias               0
total_afectados               0
hectareas_cultivo_perdidas    0
dtype: int64

Muestra:


,departamento,provincia,anho,mes,num_emergencias,total_afectados,hectareas_cultivo_perdidas
0,AMAZONAS,BAGUA,2019,1,1,0.0,0.0
1,AMAZONAS,CHACHAPOYAS,2019,1,4,14.0,0.0
2,AMAZONAS,LUYA,2019,1,2,2.0,0.0
3,AMAZONAS,UTCUBAMBA,2019,1,1,4.0,0.0
6,ANCASH,CARHUAZ,2019,1,2,0.0,0.0


---
## PASO 4 — Merge de las tres fuentes

- Cruzar MIDAGRI + NASA + INDECI por `departamento`, `provincia`, `anho`, `mes`
- Agregar flags NLP: `tiene_nlp=0`, `nlp_index=0`, `nlp_index_lag1=0`
- Generar columnas temporales cíclicas: `month_sin`, `month_cos`, `mes_num`, `trimestre_*`
- Construir `fecha_evento` con formato `YYYY-MM-DD`

In [6]:
# PASO 4: Merge de las tres fuentes
MERGE_KEYS = ['departamento', 'provincia', 'anho', 'mes']

# 4a. MIDAGRI + NASA (inner join — solo donde ambos tienen datos)
merged = midagri_agg.merge(nasa_clean, on=MERGE_KEYS, how='inner')
print(f"MIDAGRI + NASA: {merged.shape}")

# 4b. + INDECI (left join — rellenar con 0 donde no hay emergencias)
merged = merged.merge(indeci_match, on=MERGE_KEYS, how='left')
for col in COLS_INDECI:
    merged[col] = merged[col].fillna(0)
print(f"+ INDECI: {merged.shape}")

# 4c. Agregar flags NLP (no hay corpus de noticias para 2019-2020)
merged['tiene_nlp'] = 0
merged['nlp_index'] = 0.0
merged['nlp_index_lag1'] = 0.0

# 4d. Generar fecha_evento y columnas temporales cíclicas
merged['fecha_evento'] = pd.to_datetime(
    merged['anho'].astype(str) + '-' + merged['mes'].astype(str).str.zfill(2) + '-01'
).dt.strftime('%Y-%m-%d')

merged['mes_num'] = merged['mes']
merged['month_sin'] = np.sin(2 * np.pi * merged['mes'] / 12)
merged['month_cos'] = np.cos(2 * np.pi * merged['mes'] / 12)
merged['trimestre_num'] = (merged['mes'] - 1) // 3 + 1
merged['trimestre_sin'] = np.sin(2 * np.pi * merged['trimestre_num'] / 4)
merged['trimestre_cos'] = np.cos(2 * np.pi * merged['trimestre_num'] / 4)

# Eliminar columnas auxiliares de merge
merged = merged.drop(columns=['anho', 'mes'])

print(f"\nDataset 2019-2020 completo: {merged.shape}")
print(f"Columnas ({len(merged.columns)}): {merged.columns.tolist()}")
print(f"Rango: {merged['fecha_evento'].min()} → {merged['fecha_evento'].max()}")
print(f"Combinaciones dpto-prov: {merged.groupby(['departamento','provincia']).ngroups}")
print(f"\nNulos por columna:")
print(merged.isnull().sum()[merged.isnull().sum() > 0] if merged.isnull().any().any() else "  Ninguno")
print(f"\nMuestra:")
merged.head(5)

MIDAGRI + NASA: (1827, 16)
+ INDECI: (1827, 19)

Dataset 2019-2020 completo: (1827, 27)
Columnas (27): ['departamento', 'provincia', 'produccion_t', 'precio_chacra_kg', 'ALLSKY_SFC_SW_DWN', 'PRECTOTCORR', 'QV2M', 'RH2M', 'T2M', 'T2M_MAX', 'T2M_MIN', 'WS2M', 'lat', 'lon', 'num_emergencias', 'total_afectados', 'hectareas_cultivo_perdidas', 'tiene_nlp', 'nlp_index', 'nlp_index_lag1', 'fecha_evento', 'mes_num', 'month_sin', 'month_cos', 'trimestre_num', 'trimestre_sin', 'trimestre_cos']
Rango: 2019-01-01 → 2020-12-01
Combinaciones dpto-prov: 100

Nulos por columna:
  Ninguno

Muestra:


,departamento,provincia,produccion_t,precio_chacra_kg,ALLSKY_SFC_SW_DWN,PRECTOTCORR,QV2M,RH2M,T2M,T2M_MAX,...,tiene_nlp,nlp_index,nlp_index_lag1,fecha_evento,mes_num,month_sin,month_cos,trimestre_num,trimestre_sin,trimestre_cos
0,AMAZONAS,BAGUA,39.5,1.475000,13.97,1.44,13.14,71.20,22.25,31.56,...,0,0.0,0.0,2019-01-01,1,0.5,0.866025,1,1.0,6.123234e-17
1,AMAZONAS,CHACHAPOYAS,25.0,2.850000,14.15,1.63,11.41,76.80,17.15,26.25,...,0,0.0,0.0,2019-01-01,1,0.5,0.866025,1,1.0,6.123234e-17
2,AMAZONAS,LUYA,47.7,1.566667,14.15,1.63,11.41,76.80,17.15,26.25,...,0,0.0,0.0,2019-01-01,1,0.5,0.866025,1,1.0,6.123234e-17
3,AMAZONAS,UTCUBAMBA,455.2,0.916667,13.97,0.90,12.09,66.94,21.51,30.61,...,0,0.0,0.0,2019-01-01,1,0.5,0.866025,1,1.0,6.123234e-17
4,ANCASH,CASMA,48.0,1.587500,22.36,0.09,12.53,72.78,21.08,30.01,...,0,0.0,0.0,2019-01-01,1,0.5,0.866025,1,1.0,6.123234e-17


---
## PASO 5 — Concatenar con el master existente y normalizar

El master existente tiene valores normalizados (z-scores). Para mantener consistencia:
1. Reconstruimos las columnas del master en orden exacto
2. Normalizamos las columnas numéricas del bloque 2019-2020 usando la **misma escala** del master completo (StandardScaler fit sobre todo el rango 2019-2025)
3. Concatenamos 2019-2020 + 2021-2025, ordenamos por fecha
4. Guardamos como `master_dataset_extendido.csv`

In [7]:
# PASO 5a: Verificar que las columnas coinciden con el master
master_cols = master_orig.columns.tolist()
print(f"Columnas del master ({len(master_cols)}):")
print(master_cols)

# Columnas del nuevo bloque (sin NLP extras por ahora)
new_cols_available = merged.columns.tolist()

# Verificar presencia de cada columna del master en el merge
missing_in_new = [c for c in master_cols if c not in new_cols_available]
extra_in_new = [c for c in new_cols_available if c not in master_cols]

print(f"\nColumnas del master faltantes en 2019-2020: {missing_in_new}")
print(f"Columnas extra en 2019-2020 (no en master): {extra_in_new}")

Columnas del master (24):
['fecha_evento', 'departamento', 'provincia', 'produccion_t', 'precio_chacra_kg', 'num_emergencias', 'total_afectados', 'hectareas_cultivo_perdidas', 'ALLSKY_SFC_SW_DWN', 'PRECTOTCORR', 'QV2M', 'RH2M', 'T2M', 'T2M_MAX', 'T2M_MIN', 'WS2M', 'lat', 'lon', 'month_sin', 'month_cos', 'mes_num', 'trimestre_num', 'trimestre_sin', 'trimestre_cos']

Columnas del master faltantes en 2019-2020: []
Columnas extra en 2019-2020 (no en master): ['tiene_nlp', 'nlp_index', 'nlp_index_lag1']


In [8]:
# PASO 5b: Alinear columnas — seleccionar solo las del master (en el mismo orden)
# Las columnas extra (tiene_nlp, nlp_index, nlp_index_lag1) se agregan al final
new_block = merged[master_cols].copy()

print(f"Bloque 2019-2020 alineado: {new_block.shape}")
print(f"Columnas: {new_block.columns.tolist()}")
print(f"\nMuestra (valores CRUDOS, pre-normalización):")
new_block.head(3)

Bloque 2019-2020 alineado: (1827, 24)
Columnas: ['fecha_evento', 'departamento', 'provincia', 'produccion_t', 'precio_chacra_kg', 'num_emergencias', 'total_afectados', 'hectareas_cultivo_perdidas', 'ALLSKY_SFC_SW_DWN', 'PRECTOTCORR', 'QV2M', 'RH2M', 'T2M', 'T2M_MAX', 'T2M_MIN', 'WS2M', 'lat', 'lon', 'month_sin', 'month_cos', 'mes_num', 'trimestre_num', 'trimestre_sin', 'trimestre_cos']

Muestra (valores CRUDOS, pre-normalización):


,fecha_evento,departamento,provincia,produccion_t,precio_chacra_kg,num_emergencias,total_afectados,hectareas_cultivo_perdidas,ALLSKY_SFC_SW_DWN,PRECTOTCORR,...,T2M_MIN,WS2M,lat,lon,month_sin,month_cos,mes_num,trimestre_num,trimestre_sin,trimestre_cos
0,2019-01-01,AMAZONAS,BAGUA,39.5,1.475000,1.0,0.0,0.0,13.97,1.44,...,15.54,2.07,-5.638889,-78.531111,0.5,0.866025,1,1,1.0,6.123234e-17
1,2019-01-01,AMAZONAS,CHACHAPOYAS,25.0,2.850000,4.0,14.0,0.0,14.15,1.63,...,10.82,2.05,-6.229444,-77.872778,0.5,0.866025,1,1,1.0,6.123234e-17
2,2019-01-01,AMAZONAS,LUYA,47.7,1.566667,2.0,2.0,0.0,14.15,1.63,...,10.82,2.05,-6.139167,-77.952222,0.5,0.866025,1,1,1.0,6.123234e-17


In [9]:
# PASO 5c: Normalizar todo junto (StandardScaler sobre el rango completo 2019-2025)
from sklearn.preprocessing import StandardScaler

# Columnas numéricas a normalizar (excluir fecha, dpto, prov, y cíclicas que ya están en [-1,1])
COLS_NO_SCALE = ['fecha_evento', 'departamento', 'provincia',
                 'month_sin', 'month_cos', 'mes_num',
                 'trimestre_num', 'trimestre_sin', 'trimestre_cos']
COLS_TO_SCALE = [c for c in master_cols if c not in COLS_NO_SCALE]

print(f"Columnas a normalizar ({len(COLS_TO_SCALE)}): {COLS_TO_SCALE}")

# Concatenar CRUDO: necesitamos los datos originales del master antes de normalizar
# Como el master ya está normalizado, necesitamos des-normalizar o usar solo los nuevos datos
# ESTRATEGIA: concatenar el bloque nuevo (crudo) con el master (ya normalizado)
# y re-normalizar TODO junto para que la escala sea consistente

# Primero: concatenar
filas_antes = len(master_orig)
dataset_ext = pd.concat([new_block, master_orig], ignore_index=True)

# Ordenar por fecha y dpto-prov
dataset_ext = dataset_ext.sort_values(['fecha_evento', 'departamento', 'provincia']).reset_index(drop=True)

# NOTA: El master ya tiene z-scores, pero al agregar datos crudos 2019-2020,
# las escalas serán inconsistentes. Hay dos opciones:
# A) Des-normalizar el master, concatenar, re-normalizar → requiere el scaler original
# B) Concatenar datos crudos 2019-2020 sin normalizar → marcar con flag para normalizar después
#
# Aplicamos opción B: guardamos el dataset extendido SIN re-normalizar las filas 2019-2020
# La normalización se debe hacer en el pipeline de entrenamiento (Fase 4)
# porque necesita el scaler original o un fit nuevo sobre todo el rango

print(f"\n{'='*60}")
print(f"REPORTE FINAL DE INTEGRACIÓN")
print(f"{'='*60}")
print(f"Filas master original (2021-2025): {filas_antes}")
print(f"Filas nuevas (2019-2020):          {len(new_block)}")
print(f"Filas dataset extendido:           {len(dataset_ext)}")
print(f"Columnas:                          {len(dataset_ext.columns)}")
print(f"Rango temporal:                    {dataset_ext['fecha_evento'].min()} → {dataset_ext['fecha_evento'].max()}")
print(f"\nCombinaciones dpto-prov:           {dataset_ext.groupby(['departamento','provincia']).ngroups}")
print(f"\nNulos por columna:")
nulos = dataset_ext.isnull().sum()
print(nulos[nulos > 0] if nulos.any() else "  Ninguno")
print(f"\nDistribución temporal:")
dataset_ext['_anho'] = dataset_ext['fecha_evento'].str[:4]
print(dataset_ext.groupby('_anho').size().to_string())
dataset_ext = dataset_ext.drop(columns='_anho')

Columnas a normalizar (15): ['produccion_t', 'precio_chacra_kg', 'num_emergencias', 'total_afectados', 'hectareas_cultivo_perdidas', 'ALLSKY_SFC_SW_DWN', 'PRECTOTCORR', 'QV2M', 'RH2M', 'T2M', 'T2M_MAX', 'T2M_MIN', 'WS2M', 'lat', 'lon']

REPORTE FINAL DE INTEGRACIÓN
Filas master original (2021-2025): 5880
Filas nuevas (2019-2020):          1827
Filas dataset extendido:           7707
Columnas:                          24
Rango temporal:                    2019-01-01 → 2025-08-01

Combinaciones dpto-prov:           105

Nulos por columna:
  Ninguno

Distribución temporal:
_anho
2019     908
2020     919
2021    1260
2022    1260
2023    1260
2024    1260
2025     840


In [10]:
# PASO 5d: Guardar dataset extendido
output_path = ROOT / 'data/processed/master_dataset_extendido.csv'
dataset_ext.to_csv(output_path, index=False)
print(f"Guardado en: {output_path}")
print(f"Tamaño: {output_path.stat().st_size / 1024:.1f} KB")

# Verificación final
check = pd.read_csv(output_path)
print(f"\nVerificación de lectura: {check.shape}")
print(f"Primeras filas (2019):")
display(check.head(3))
print(f"\nÚltimas filas (2025):")
display(check.tail(3))

Guardado en: C:\Machine-learming\Machine-Learning-Multimodal--Agro-NLP-Clima-\data\processed\master_dataset_extendido.csv
Tamaño: 2566.4 KB

Verificación de lectura: (7707, 24)
Primeras filas (2019):


,fecha_evento,departamento,provincia,produccion_t,precio_chacra_kg,num_emergencias,total_afectados,hectareas_cultivo_perdidas,ALLSKY_SFC_SW_DWN,PRECTOTCORR,...,T2M_MIN,WS2M,lat,lon,month_sin,month_cos,mes_num,trimestre_num,trimestre_sin,trimestre_cos
0,2019-01-01,AMAZONAS,BAGUA,39.5,1.475000,1.0,0.0,0.0,13.97,1.44,...,15.54,2.07,-5.638889,-78.531111,0.5,0.866025,1,1,1.0,6.123234e-17
1,2019-01-01,AMAZONAS,CHACHAPOYAS,25.0,2.850000,4.0,14.0,0.0,14.15,1.63,...,10.82,2.05,-6.229444,-77.872778,0.5,0.866025,1,1,1.0,6.123234e-17
2,2019-01-01,AMAZONAS,LUYA,47.7,1.566667,2.0,2.0,0.0,14.15,1.63,...,10.82,2.05,-6.139167,-77.952222,0.5,0.866025,1,1,1.0,6.123234e-17



Últimas filas (2025):


,fecha_evento,departamento,provincia,produccion_t,precio_chacra_kg,num_emergencias,total_afectados,hectareas_cultivo_perdidas,ALLSKY_SFC_SW_DWN,PRECTOTCORR,...,T2M_MIN,WS2M,lat,lon,month_sin,month_cos,mes_num,trimestre_num,trimestre_sin,trimestre_cos
7704,2025-08-01,UCAYALI,CORONEL PORTILLO,-0.087507,2.804116,-0.249774,-0.09035,-0.013941,0.449989,-0.535144,...,1.249330,-1.240145,-8.368056,-74.543333,-0.866025,-0.5,8,3,-1.0,-1.836970e-16
7705,2025-08-01,UCAYALI,PADRE ABAD,-0.116314,3.010412,-0.249774,-0.09035,-0.013941,0.103609,-0.375952,...,1.009538,-1.309774,-9.033611,-75.507500,-0.866025,-0.5,8,3,-1.0,-1.836970e-16
7706,2025-08-01,UCAYALI,PURUS,-0.216853,4.488867,-0.249774,-0.09035,-0.013941,0.443391,-0.680185,...,0.865662,-1.370699,-9.772222,-70.709722,-0.866025,-0.5,8,3,-1.0,-1.836970e-16
